# Framework Microsoft Agent — Azure OpenAI (API de réponses)

Dans cet exemple de code, vous utiliserez le **Microsoft Agent Framework (MAF)** pour créer un agent simple soutenu par **Azure OpenAI** en utilisant l'**API de réponses**.

> **Note de migration :** Cet exemple utilisait auparavant Semantic Kernel avec GitHub Models. Il a été migré vers le Microsoft Agent Framework, et GitHub Models (obsolète, retiré en juillet 2026) a été remplacé par Azure OpenAI, qui prend en charge l'API de réponses. Le `OpenAIChatClient` dans MAF cible le point de terminaison stable `/openai/v1/` d'Azure OpenAI et utilise par défaut l'API de réponses.

Le but de cet exemple est de démontrer les étapes qui seront ensuite appliquées dans d'autres exemples de code lors de l'implémentation de différents modèles agentiques.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Importer les packages Python nécessaires


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Définition d'un outil

Dans le Microsoft Agent Framework, un **outil** est une fonction Python simple décorée avec `@tool` que l'agent peut appeler. Ci-dessous, nous définissons un outil qui renvoie une destination de vacances aléatoire et évite de répéter la précédente.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Création de l'Agent

Ici, nous créons l'Agent nommé `TravelAgent`.

Dans cet exemple, nous utilisons des instructions très basiques. N'hésitez pas à modifier ces instructions pour observer comment le comportement de l'agent change.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Exécution de l’agent

Nous pouvons maintenant exécuter l’agent. Nous créons une `AgentSession` pour que l’agent se souvienne de la conversation au fil des échanges, puis envoyons deux `user_inputs`. Le premier demande un voyage ; le second indique que l’utilisateur n’a pas aimé la suggestion et en demande une autre — l’agent utilise l’historique de session ainsi que l’outil `get_random_destination` pour répondre.

Vous pouvez modifier ces messages pour observer comment l’agent réagit différemment. Les réponses sont **diffusées** token par token.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
